<h1 align="center">Laboratorio 10</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab10)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab10.ipynb --to html

## Task 1

En clase explicamos que la Difusión "esculpe" la imagen restando ruido de un tensor dentro del Espacio Latente, paso a paso, antes de que el Decoder la convierta en píxeles. Usted deberá demostrar este comportamiento interrumpiendo el proceso para observar la evolución temporal.

Para esto considere las siguientes instrucciones paso a paso:

1. Instancie un modelo estándar de difusión (por ejemplo, StableDiffusionPipeline usando los pesos de la versión 1.5). Asegúrese de enviar el modelo a la GPU (cuda).
2. Defina exactamente 20 pasos de inferencia (num_inference_steps=20) y fije una semilla aleatoria (Seed) usando torch.manual_seed() para garantizar la reproducibilidad.
3. Utilice un prompt que exija al modelo generar estructuras geométricas y texturas complejas. Ejemplo obligatorio: "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution".
4. Usted no debe generar la imagen final de un solo golpe. Su código debe interceptar el tensor latente (matriz matemática de ruido) durante el loop de denoising.
   1. Hint: Para lograr esto sin escribir el pipeline desde cero, investigue en la documentación de Hugging Face el uso de la función callback_on_step_end dentro de la llamada al pipeline, o bien, desempaquete el pipeline y escriba el ciclo for manualmente usando el scheduler.step().
5. Guarde una copia del tensor latente en memoria exactamente en los pasos 4, 10, 16 y 20.
6. Tome esos 4 tensores latentes y páselos manualmente por el componente decodificador de su modelo (vae.decode()).
   1. Warning: Recuerde que los modelos latentes escalan matemáticamente los tensores. Antes de pasar su tensor al VAE, investigue si debe multiplicarlo o dividirlo por el scaling_factor del VAE para evitar errores de dimensión o imágenes en negro.
7. Convierta la salida del VAE a formato de imagen (PIL) y guárdelas.

De esta parte se espera que entregue y responda:

- Muestre la cuadrícula visual con la evolución de las 4 imágenes.
- Observe la diferencia entre la imagen del paso 4 y la del paso 16. Explicado con base en la teoría de frecuencias espaciales y Cross-Attention: ¿Qué características de la imagen (forma global, colores bases, silueta vs. brillos, texturas finas, detalles del neón) resuelve la U-Net en las etapas iniciales de ruido alto, y qué resuelve en las etapas finales de ruido bajo? Justifique técnicamente su respuesta.

### Requisitos

- Python 3.10 o superior (se utilizó 3.12)
- GPU NVIDIA con al menos 6 GB de VRAM (en este equipo: RTX 5070, 12 GB, CUDA 13.2)
- Conexion a Internet en la primera ejecucion para descargar los pesos de SD 1.5 (~4 GB) desde Hugging Face Hub

### Estructura

- `task1.py` – script principal
- `requirements.txt` – dependencias con versiones fijadas
- `outputs/` – se genera al ejecutar (`step_04.png`, `step_10.png`, `step_16.png`, `step_20.png`, `grid.png`)

### Instalación

```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128
pip install -r requirements.txt
```

#### Verificación

```bash
python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0)); print(torch.cuda.get_arch_list())"
```

Salida esperada: `True NVIDIA GeForce RTX 5070` y la lista de arquitecturas debe incluir `sm_120` (o superior).

### Uso

#### Base

`python task1.py [--output-dir OUT] [--seed N] [--model MODEL_ID]`

#### Ejecucion

```bash
python task1.py
```

Opciones:

```bash
python task1.py --output-dir outputs --seed 42 --model runwayml/stable-diffusion-v1-5
```

### Salida

Por defecto en ./outputs:

`step_04.png, step_10.png, step_16.png, step_20.png, grid.png`



### Decisiones tecnicas

#### Captura del latente

Se usa el parametro `callback_on_step_end` del pipeline de diffusers (introducido en 0.27+). Diffusers indexa los pasos desde 0; el callback convierte a base 1 antes de comparar con `{4, 10, 16, 20}`. El latente se clona y se guarda **desconectado del grafo** para no acumular memoria de gradientes.

Alternativa equivalente seria desempaquetar el `for` del pipeline y llamar manualmente a `scheduler.step()` y `unet(...)`, pero el callback es mas limpio y mantiene el resto de la maquinaria del pipeline (CFG, embeddings de texto, etc.) intacta.

#### Escalado del latente antes del VAE

Durante el denoising, los latentes viven en el espacio escalado por `vae.config.scaling_factor` (en SD 1.5 vale 0.18215). El decoder espera latentes en el rango sin escalar, por lo que antes de `vae.decode()` se aplica:

```python
latent = latent / vae.config.scaling_factor
```

Si se omite, las imagenes salen completamente negras o saturadas. La operacion inversa (multiplicar) es la que aplica el encoder al codificar una imagen real para image-to-image; aqui vamos en sentido contrario y por eso se divide.

#### Conversion a PIL

La salida del decoder esta en rango $[-1, 1]$. Se reescala a $[0, 1]$, se mueve a CPU, se convierte a `uint8` en rango $[0, 255]$ y se construye la imagen PIL. Esto se hace dentro de `torch.no_grad()` para evitar reservar memoria de gradientes innecesaria.

### Respuesta teorica

#### Diferencia entre el paso 4 y el paso 16

En el **paso 4** (apenas 20 % del proceso de denoising completado), la imagen decodificada se ve como estatica cromatica de alta frecuencia, sin silueta ni composicion reconocible. Domina un patron de confeti multicolor en azules, rojos, amarillos y blancos distribuido de forma cuasi-uniforme. Esto ocurre porque el latente todavia esta dominado por el ruido $\varepsilon$ y el VAE, que fue entrenado exclusivamente sobre latentes limpios, **no sabe mapear ese tensor a una imagen coherente**: lo que decodifica son los rasgos de bajo nivel del ruido residual, no contenido semantico.

En el **paso 16** (80 % completado), la composicion global ya esta resuelta. Se distingue claramente una vitrina horizontal con frutas y verduras al frente (rojos, naranjas, amarillos y verdes formando hileras coherentes) y una estanteria con elementos tubulares al fondo que se lee como un ambiente industrial frio. La separacion figura-fondo es nitida y los objetos principales tienen forma reconocible. Aun quedan artefactos en zonas grandes del fondo y los bordes no son perfectamente limpios, pero el "que" de la imagen esta definitivamente fijado.

#### Justificacion tecnica: frecuencias espaciales y Cross-Attention

El comportamiento se explica por la interaccion entre el calendario de ruido (noise schedule) y el contenido en frecuencias espaciales que la U-Net puede recuperar en cada momento.

**1. Pasos iniciales (alto $\sigma$, bajo SNR): bajas frecuencias.**

Cuando el ruido inyectado domina sobre la señal (relacion señal-ruido baja, $\bar{\alpha}_t$ pequeño), las altas frecuencias (texturas finas, bordes agudos, microdetalles) estan sepultadas por el ruido gaussiano, que es por construccion plano en el espectro. Solo las bajas frecuencias (gradientes de color amplios, distribucion global de luz, silueta general) sobreviven con suficiente SNR para que la U-Net pueda inferir algo significativo de ellas.

En estos pasos la U-Net resuelve composicion global: donde va el objeto, paleta general, contraste figura-fondo. Es coherente con la teoria: predecir un detalle fino a partir de puro ruido es indeterminado, pero predecir "esta zona tiende a tonos calidos y aquella a tonos frios" es estadisticamente posible. La razon por la que en el paso 4 no vemos esa "version borrosa" idealizada es que **el VAE no esta diseñado para decodificar latentes ruidosos**; el trabajo de bajas frecuencias ya esta ocurriendo dentro del latente, pero solo se hace visible cuando el ruido cae lo suficiente para que el decoder pueda interpretarlo (alrededor del paso 12-14 en este caso).

**2. Pasos finales (bajo $\sigma$, alto SNR): altas frecuencias.**

A medida que avanza el denoising y $\bar{\alpha}_t$ se aproxima a 1, la imagen subyacente domina sobre el ruido residual. Ahora las altas frecuencias emergen del fondo de ruido y la U-Net puede dedicar su capacidad a refinar bordes de cada fruta, texturas de cascara y piel, brillos especulares en la vitrina, microcontraste entre productos contiguos y reflejos sub-pixel en las superficies metalicas. La composicion global ya esta fija desde los primeros pasos, asi que estos pasos finales solo pulen detalle.

Este comportamiento se conoce como **coarse-to-fine generation** y es la razon por la que las muestras de difusion se ven "borrosas pero correctas" en pasos intermedios y "nitidas y consistentes" al final.

**3. Rol del Cross-Attention con CLIP.**

El Cross-Attention opera en cada bloque de la U-Net, pero su efecto no es uniforme a lo largo del proceso:

- En los pasos iniciales, los queries $Q$ vienen de un latente casi puro ruido. Las activaciones que matchean keys $K$ del prompt CLIP son las que disparan **patrones gruesos asociados a conceptos**; la palabra "fruit" activa una distribucion espacial de objetos pequeños y redondeados agrupados en una franja; "laboratory" empuja hacia un ambiente arquitectonico cerrado con elementos tecnicos al fondo; "cyberpunk" y "neon lights" sesgan la paleta hacia contraste alto y luces puntuales. Esto se manifiesta a nivel de latente como una **distribucion espacial gruesa de regiones semanticas** que el decoder solo logra interpretar varios pasos despues.
- En los pasos finales, los queries provienen de un latente con estructura ya formada. El Cross-Attention refuerza **alineamiento semantico fino**; "glowing" empuja brillos especulares concretos sobre las frutas, "highly detailed" y "4k" sesgan la red hacia altas frecuencias y texturas crujientes, "fruit" termina de definir formas reconocibles (manzanas, citricos, hojas) en lugar de manchas calidas. El texto deja de definir composicion y pasa a **dictar textura y estilo**.

Es interesante notar que el modelo interpreto "cyberpunk laboratory" no como neones magenta y violeta saturados, sino como una vitrina refrigerada de aspecto industrial con tubos y luz fria. Esto refleja como CLIP balancea conceptos potencialmente conflictivos del prompt: "fruit" arrastro la composicion hacia una escena de exhibicion comercial, y "cyberpunk laboratory" se materializo en la estetica del fondo (tubos, equipo industrial) en lugar de invadir el primer plano.

**4. Conexion con el calendario de Markov.**

El calendario $\beta_t$ (lineal o cosenoidal) esta diseñado precisamente para que la mayor parte del trabajo de composicion ocurra en los primeros pasos (cuando el ruido cae rapido) y los pasos finales se dediquen a refinamiento de alta frecuencia con incrementos pequeños. Por eso 20 pasos bastan para obtener una imagen reconocible: los pasos 1 a 6 fijan el "que" (aunque el VAE aun no pueda decodificarlo limpiamente), los pasos 10 a 14 vuelven la composicion legible, y los pasos 15 a 20 fijan el "como se ve en detalle".

#### Sintesis

| Etapa | Ruido relativo | Que esta resolviendo realmente la U-Net | Que se ve al decodificar | Que aporta el Cross-Attention |
|---|---|---|---|---|
| Paso 4 | Muy alto | Bajas frecuencias: distribucion espacial de regiones semanticas | Estatica multicolor (VAE no puede leer latente ruidoso) | Anclaje semantico grueso ("fruit", "laboratory", paleta calida vs fria) |
| Paso 16 | Bajo | Altas frecuencias: bordes, texturas, brillos | Vitrina con frutas/verduras y fondo industrial reconocibles | Refinamiento estilistico ("glowing", "highly detailed", "4k") |

La diferencia visual entre los pasos 4 y 16 es la materializacion directa del principio coarse-to-fine de la difusion: **primero la idea, luego el detalle**. Que el paso 4 no se vea borroso sino caotico es ademas una evidencia adicional del rol del VAE como decoder especializado en latentes limpios, no en latentes intermedios del proceso.


### Instalación y Validación

![WSL 2 - Python 3.12](./images/t1-p0.png)

![WSL 2 - Python 3.12](./images/t1-p1.png)

![WSL 2 - Python 3.12](./images/t1-p2.png)

### Resultados

![Grid](./outputs/grid.png)

![Step 4](./outputs/step_04.png)

![Step 10](./outputs/step_10.png)

![Step 16](./outputs/step_16.png)

![Step 20](./outputs/step_20.png)

## Task 2

Como vimos al final de la sesión, el modelo Nano Banana de Google representa la vanguardia en eficiencia: usar técnicas de destilación para saltarse pasos matemáticos de la Cadena de Markov y correr modelos generativos en milisegundos. Usted simulará este escenario midiendo empíricamente el trade-off (costo-beneficio) en producción.

Para esto considere las siguientes instrucciones paso a paso:

1. Escenario A (Modelo Estándar - Costoso):

   1. Utilice el modelo estándar de la Sección 1.
   2. Configure la generación a 50 pasos de inferencia.
   3. Use la misma semilla y el mismo prompt.
   4. Implemente medidores de rendimiento en su código: use time.time() para medir los segundos exactos que tarda la inferencia, y torch.cuda.max_memory_allocated() para medir el pico de VRAM consumida.

2. Escenario B (Modelo Destilado - Eficiente):

   1. Para simular el paradigma de "Nano Banana", usted debe cargar una arquitectura diseñada para pocos pasos (Destilación).
   2. Investigue e instancie un modelo basado en Turbo o LCM. (Ejemplo: SDXL-Turbo, SD-Turbo, o cargar pesos LCM en su modelo base).
   3. Configure la generación a únicamente 4 pasos de inferencia.
   4. Mida y registre el tiempo de ejecución y el consumo máximo de VRAM de la misma manera que en el Escenario A.

De esta parte se espera que entregue y responda:

- Presente las dos imágenes resultantes lado a lado.
- Muestre una tabla comparando: Modelo usado, Pasos, Tiempo de Ejecución (segundos) y VRAM (MB/GB).
- Como Arquitecto de IA encargado de desplegar esta API en una aplicación con millones de usuarios:
  - Describa brevemente cómo la técnica de "Destilación" permitió al modelo B generar una imagen coherente en solo 4 pasos, mientras que si usted pusiera el modelo A a 4 pasos el resultado sería ruido inservible.
  - Analice sus métricas (Tiempo y Memoria). Con base en la calidad visual obtenida frente al ahorro de hardware, emita un dictamen justificando cuál de los dos escenarios elegiría para producción y por qué.

Recuerden: Las respuestas vagas no sumarán puntos. Se busca que demuestre dominio técnico del flujo de datos (Tensores → U-Net → VAE → Pixeles) y capacidad de toma de decisiones arquitectónicas en ecosistemas de Inteligencia Artificial.

## Referencias

- [Stable Diffusion 1.5 (mirror oficial en Hugging Face)](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)
- [diffusers – Callbacks API](https://huggingface.co/docs/diffusers/using-diffusers/callback)
- [diffusers – AutoencoderKL](https://huggingface.co/docs/diffusers/api/models/autoencoderkl)
- [High-Resolution Image Synthesis with Latent Diffusion Models (Rombach et al., 2022)](https://arxiv.org/abs/2112.10752)
- [Elucidating the Design Space of Diffusion-Based Generative Models (Karras et al., 2022)](https://arxiv.org/abs/2206.00364)